# Lab 19–22b — Quantization error, measured

**Two questions.** How much error does a `b`-bit quantizer add, and how does it grow as you remove bits?
And when does that error start to cost a real model its accuracy?

| Part | You quantize | You measure |
| --- | --- | --- |
| **1. The error playground** | a sine wave and Gaussian noise, at 1–8 bits | error vs step size, signal-to-noise ratio per bit, the cost of one outlier |
| **2. A small model** | the weights of a digit classifier, at 1–8 bits | test accuracy, per-tensor vs per-channel |

Read the [lab brief](Lab19_22b.md) first. Marked on **relationships between your own numbers**; your signal
and your model are seeded from your roll number, so your numbers are yours.

**Before you run anything:** set your identity below. Fill every 📝 cell. Run top to bottom, then run the
final export cell and submit the two files it names. Runs in under a minute on any CPU.

In [ ]:
ROLL_NUMBER = ""      # <- your roll number, e.g. "202512345"
NAME        = ""      # <- your name

assert ROLL_NUMBER and NAME, "Set ROLL_NUMBER and NAME before running the rest."

In [ ]:
import json, platform, sys, hashlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import sklearn

SEED = int(hashlib.sha256(ROLL_NUMBER.encode()).hexdigest(), 16) % (2**31)   # your seed
rng = np.random.default_rng(SEED)
BITS = list(range(1, 9))
RESULTS = {}

---
## Part 1 — The error playground

### The quantizer

Lecture 21/22's affine map, on the **unsigned** grid `[0, 2^b − 1]`, with one `(S, Z)` for the whole array:

```
stretch:   r_min = min(x.min(), 0),  r_max = max(x.max(), 0)      # 0.0 must be inside the range (§2)
scale:     S = (r_max − r_min) / (2^b − 1)                         # Equation 2.5
zero:      Z = round(−r_min / S)                                   # Equation 2.7, with q_min = 0
quantize:  q = clip(round(x / S + Z), 0, 2^b − 1)                  # Equation 2.4
decode:    x̂ = S (q − Z)                                           # Equation 2.2
```

### Two ways to measure the error

- **Mean squared error**, `MSE = mean((x̂ − x) ** 2)`. Rounding moves each value by at most `S/2`, and for
  a busy signal the moves are spread evenly across `[−S/2, S/2]`. The mean square of an even spread over a
  width `S` is `S² / 12`, so **`MSE ≈ S² / 12`**.
- **Signal-to-noise ratio**, in decibels: `SNR = 10 · log10( sum(x ** 2) / sum((x − x̂) ** 2) )`. It is
  the signal's power over the error's power. Every 10 dB is a factor of 10 in power.

In [ ]:
def quantize(x, bits):
    '''x: numpy array. Return (q, S, Z): the integer array q, and the scale and zero-point it was
       quantized with, using one (S, Z) for the whole of x. Follow the five lines above.'''
    # TODO: your code here
    raise NotImplementedError

def dequantize(q, S, Z):
    '''Decode the integers back to real values.'''
    # TODO: your code here
    raise NotImplementedError

def snr_db(x, x_hat):
    '''Signal-to-noise ratio of the approximation x_hat, in dB.'''
    # TODO: your code here
    raise NotImplementedError


# given: your two signals, 4096 samples each
N = 4096
t = np.arange(N) / N
CYCLES = 3 + SEED % 7
SIGNALS = {"sine":  np.sin(2 * np.pi * CYCLES * t + rng.uniform(0, 2 * np.pi)),
           "noise": rng.normal(0.0, 1.0, N)}
for name, x in SIGNALS.items():
    print(f"{name:6s} min {x.min():6.3f}   max {x.max():6.3f}   mean square {np.mean(x ** 2):.3f}")

### 1a — Look at one quantizer

At 3 bits the grid has 8 levels. The plot shows the sine, its quantized version, and where the 8 levels sit
over the histogram of the sine's values.

📝 **Predict.** For the sine (range about −1 to 1), what is `S` at 3 bits? At 8 bits? Each extra bit does
what to `S`, and so what to `S² / 12`?

*(your predictions here)*

In [ ]:
# given: one quantizer, drawn
x = SIGNALS["sine"]
q, S, Z = quantize(x, 3)
x_hat = dequantize(q, S, Z)
levels = S * (np.arange(2 ** 3) - Z)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].plot(t[:600], x[:600], label="sine"); ax[0].step(t[:600], x_hat[:600], where="mid", label="3-bit")
ax[0].set_title("signal and its 3-bit version"); ax[0].legend()
ax[1].hist(x, bins=80, color="0.7"); [ax[1].axvline(l, color="C1") for l in levels]
ax[1].set_title("sine values, with the 8 levels"); plt.tight_layout(); plt.show()
print(f"3 bits: S = {S:.4f}, Z = {Z:.0f}, levels = {np.round(levels, 3)}")

### 1b — Sweep the bits

📝 **Predict.** By how many dB should the SNR rise for each extra bit? (One bit halves `S`; what does
halving `S` do to the error power?) Which signal, sine or noise, will have the higher SNR at the same bit
width, and why? (Look at their min and max next to their mean square.)

*(your predictions here)*

In [ ]:
# harness (given): MSE against S^2 / 12, and SNR, for both signals at every bit width
sweep = {}
for name, x in SIGNALS.items():
    rows = []
    for b in BITS:
        q, S, Z = quantize(x, b)
        x_hat = dequantize(q, S, Z)
        rows.append(dict(bits=b, S=float(S), mse=float(np.mean((x_hat - x) ** 2)),
                         s2_over_12=float(S ** 2 / 12), snr=float(snr_db(x, x_hat))))
    slope = float(np.polyfit([r["bits"] for r in rows if r["bits"] >= 3],
                             [r["snr"] for r in rows if r["bits"] >= 3], 1)[0])
    sweep[name] = dict(rows=rows, slope_db_per_bit=slope)
    print(f"\n{name}: SNR rises {slope:.2f} dB per bit (fitted over 3-8 bits)")
    for r in rows:
        print(f"  {r['bits']} bits   S {r['S']:.5f}   MSE {r['mse']:.3e}   S^2/12 {r['s2_over_12']:.3e}"
              f"   ratio {r['mse'] / r['s2_over_12']:.2f}   SNR {r['snr']:5.1f} dB")

plt.figure(figsize=(6, 3.2))
for name in SIGNALS:
    plt.plot(BITS, [r["snr"] for r in sweep[name]["rows"]], "o-", label=name)
plt.xlabel("bits"); plt.ylabel("SNR (dB)"); plt.legend(); plt.grid(alpha=.3); plt.show()
RESULTS["sweep"] = sweep

📝 **Explain.** Where does `MSE ≈ S² / 12` hold, and at which bit widths does it break down, and why? What
is the dB-per-bit rule you measured, in one sentence? Why does the noise lose to the sine at every width?

*(your answer here)*

### 1c — One outlier

Lecture 21/22 §5: one large value sets `S` for everything that shares it. The harness sets **one** sample of
the sine to 20 and quantizes at 8 bits twice: once with one `(S, Z)` for the whole signal, and once in
**blocks of 64 samples**, each with its own `(S, Z)`. It measures the SNR of every sample **except** the
outlier.

📝 **Predict.** The range grows from about 2 to about 21. By how many dB should the SNR of the other 4,095
samples drop? (`S` grows with the range; use your dB-per-bit rule.) What will blocks of 64 recover?

*(your predictions here)*

In [ ]:
# harness (given): one outlier, per-signal vs per-block scales, 8 bits
x = SIGNALS["sine"].copy()
spike = int(rng.integers(N)); x[spike] = 20.0
rest = np.arange(N) != spike

clean_hat = dequantize(*quantize(SIGNALS["sine"], 8))
whole_hat = dequantize(*quantize(x, 8))
block_hat = np.concatenate([dequantize(*quantize(x[i:i + 64], 8)) for i in range(0, N, 64)])

range_ratio = (max(x.max(), 0) - min(x.min(), 0)) / (max(SIGNALS["sine"].max(), 0) - min(SIGNALS["sine"].min(), 0))
outlier = dict(snr_clean=float(snr_db(SIGNALS["sine"], clean_hat)),
               snr_whole=float(snr_db(x[rest], whole_hat[rest])),
               snr_blocks=float(snr_db(x[rest], block_hat[rest])),
               range_ratio=float(range_ratio))
outlier["drop_db"] = outlier["snr_clean"] - outlier["snr_whole"]
print(f"8 bits, no outlier:              SNR {outlier['snr_clean']:5.1f} dB")
print(f"8 bits, outlier, one (S, Z):     SNR of the other samples {outlier['snr_whole']:5.1f} dB"
      f"   (drop {outlier['drop_db']:.1f} dB, range grew {range_ratio:.1f}x)")
print(f"8 bits, outlier, blocks of 64:   SNR of the other samples {outlier['snr_blocks']:5.1f} dB")
RESULTS["outlier"] = outlier

📝 **Explain.** Turn your drop into bits: how many of the 8 bits did one outlier waste? Why do blocks of 64
get almost all of them back, and what do the blocks cost in storage (Lecture 21/22 §5's bits-per-weight
formula, with an fp16 `S` and `Z` per block)?

*(your answer here)*

---
## Part 2 — Quantize a small model

### The model

A one-hidden-layer network that reads an 8×8 handwritten digit (64 pixels) and outputs one of 10 classes.
The cell trains it in about a second; after that it is **only two weight matrices and two bias vectors**:

```
hidden = relu(x @ W1 + b1)       # W1: (64, 64)
logits = hidden @ W2 + b2        # W2: (64, 10)
predicted digit = argmax(logits)
```

`W1[:, j]`, **column `j`**, holds the 64 weights that produce hidden unit `j`. A column is one **output
channel** (scikit-learn stores weights as `(in, out)`).

You quantize **only the weights**; the biases stay in float. Nothing is retrained.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X, y = load_digits(return_X_y=True)
X = X / 16.0                                     # pixels 0..16 -> 0..1
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=SEED % 10_000)
clf = MLPClassifier(hidden_layer_sizes=(64,), max_iter=600, random_state=SEED % 10_000).fit(X_tr, y_tr)
(W1, W2), (b1, b2) = clf.coefs_, clf.intercepts_

def accuracy(W1, W2):
    hidden = np.maximum(X_te @ W1 + b1, 0)
    return float(np.mean((hidden @ W2 + b2).argmax(1) == y_te))

FLOAT_ACC = accuracy(W1, W2)
print(f"W1 {W1.shape}, W2 {W2.shape} | float test accuracy {FLOAT_ACC:.3f} "
      f"(sklearn's own: {clf.score(X_te, y_te):.3f}) on {len(y_te)} test digits")
for name, w in [("W1", W1), ("W2", W2)]:
    print(f"{name}: range [{w.min():.3f}, {w.max():.3f}], 99% of |w| below {np.percentile(np.abs(w), 99):.3f}")

### 2a — Per-tensor vs per-channel

**Per-tensor** uses your `quantize` on a whole matrix: one `(S, Z)`. **Per-channel** gives every column its
own `(S, Z)` (Lecture 21/22 §5).

📝 **Predict.** At 8 bits, how much accuracy do you expect to lose? Below how many bits do you expect it to
fall apart? Will per-channel help, and at which widths will you be able to see it?

*(your predictions here)*

In [ ]:
def quantize_per_channel(w, bits):
    '''w: (in, out) weight matrix. Quantize each column with its own (S, Z) using your quantize, decode it,
       and return the decoded matrix, same shape as w.'''
    # TODO: your code here
    raise NotImplementedError


# harness (given): accuracy and weight SNR at every bit width, both granularities
def per_tensor(w, bits): return dequantize(*quantize(w, bits))

model = dict(float_acc=FLOAT_ACC, sklearn_acc=float(clf.score(X_te, y_te)), tensor={}, channel={})
for gran, fn in [("tensor", per_tensor), ("channel", quantize_per_channel)]:
    for b in BITS:
        W1h, W2h = fn(W1, b), fn(W2, b)
        model[gran][b] = dict(acc=accuracy(W1h, W2h), snr_w1=float(snr_db(W1, W1h)),
                              max_levels_per_column=int(max(len(np.unique(W1h[:, j])) for j in range(W1.shape[1]))))
print(" bits   per-tensor acc  (W1 SNR)    per-channel acc  (W1 SNR)")
for b in BITS:
    t_, c_ = model["tensor"][b], model["channel"][b]
    print(f"  {b}        {t_['acc']:.3f}     ({t_['snr_w1']:5.1f} dB)       {c_['acc']:.3f}     ({c_['snr_w1']:5.1f} dB)")

plt.figure(figsize=(6, 3.2))
for gran in ("tensor", "channel"):
    plt.plot(BITS, [model[gran][b]["acc"] for b in BITS], "o-", label=f"per-{gran}")
plt.axhline(FLOAT_ACC, color="0.5", ls="--", label="float")
plt.xlabel("bits"); plt.ylabel("test accuracy"); plt.legend(); plt.grid(alpha=.3); plt.show()
RESULTS["model"] = model

📝 **Explain.** Where did accuracy fall apart, and why does per-channel hold on longer? Use the printed
ranges: which columns set the per-tensor `S`?

*(your answer here)*

### 2b — Connect the two parts

📝 **Explain (the paragraph that carries this lab).** At 3 bits the weights' SNR is low, a level you would
call badly distorted for the sine in Part 1, yet the classifier barely notices. Why can a model tolerate
error that a signal metric calls large? (Think about what the model's output is: an `argmax`.) What does
that say about judging a quantized model by its weight error alone, and what should you measure instead?

*(your answer here)*

---
## Submit

Run this last. It writes `submission_lab19_22b_<roll>.json`. Submit **two files**: that JSON and this
executed notebook with every 📝 cell filled.

In [ ]:
env = dict(platform=platform.platform(), python=sys.version.split()[0],
           numpy=np.__version__, sklearn=sklearn.__version__, seed=SEED, cycles=CYCLES)
sub = dict(roll=ROLL_NUMBER, name=NAME, env=env, results=RESULTS)
out = Path(f"submission_lab19_22b_{ROLL_NUMBER}.json")
out.write_text(json.dumps(sub, indent=2))
print("wrote", out, "| parts:", list(RESULTS))
assert set(RESULTS) >= {"sweep", "outlier", "model"}, "run every part before exporting"